# ASTR 223 Final Project: GP Projections

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pybaseball.statcast as pyb
from pathlib import Path
import dask.dataframe as dd

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [2]:
data = dd.read_parquet('files/*.parquet')
statcast_data = data.query('game_type == "R"').reset_index(drop=True)
statcast_dask_data = statcast_data.compute()
statcast_dask_data

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,FC,2018-10-01,92.2,-1.97,6.26,"Jansen, Kenley",467827,445276,strikeout,swinging_strike,<NA>,<NA>,<NA>,<NA>,12,Gerardo Parra strikes out swinging.,R,L,R,LAD,COL,S,2,<NA>,0,2,2018,0.39,1.34,0.633962,3.524498,<NA>,<NA>,<NA>,2,9,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,5.848045,-134.451705,-4.262857,3.716224,25.412307,-14.775673,3.32,1.51,<NA>,<NA>,<NA>,94.1,2629,7.0,570335,518735,641355,571771,457759,592518,592626,621035,624577,53.52,<NA>,0.0,0.0,1,0,0,<NA>,71,4,Cutter,5,2,2,5,2,5,2,5,Standard,Standard,164,0.003,-0.153,<NA>,<NA>,<NA>,0.153,<NA>,3,-3,0.997,0.003,30,31,31,31,1,0,2,2,4,1,1.24,-0.39,0.39,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,FC,2018-10-01,93.0,-1.77,6.3,"Jansen, Kenley",467827,445276,<NA>,foul,<NA>,<NA>,<NA>,<NA>,12,Foul,R,L,R,LAD,COL,S,<NA>,<NA>,0,2,2018,0.52,1.26,1.088134,2.441313,<NA>,<NA>,<NA>,2,9,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,6.244121,-135.416081,-7.075826,5.344581,27.081217,-15.009031,3.32,1.51,<NA>,<NA>,<NA>,94.6,2686,7.0,570335,518735,641355,571771,457759,592518,592626,621035,624577,53.5,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,71,3,Cutter,5,2,2,5,2,5,2,5,Standard,Standard,158,0.0,0.0,<NA>,<NA>,<NA>,0.0,<NA>,3,-3,0.997,0.003,30,31,31,31,1,0,2,2,4,1,1.3,-0.52,0.52,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,FC,2018-10-01,91.6,-1.75,6.22,"Jansen, Kenley",467827,445276,<NA>,called_strike,<NA>,<NA>,<NA>,<NA>,5,Called Strike,R,L,R,LAD,COL,S,<NA>,<NA>,0,1,2018,0.64,1.14,-0.172991,2.411909,<NA>,<NA>,<NA>,2,9,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2.602442,-133.526432,-6.436552,7.337142,26.60837,-16.953719,3.36,1.68,<NA>,<NA>,<NA>,93.4,2581,7.1,570335,518735,641355,571771,457759,592518,592626,621035,624577,53.4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,71,2,Cutter,5,2,2,5,2,5,2,5,Standard,Standard,151,0.0,-0.065,<NA>,<NA>,<NA>,0.065,<NA>,3,-3,0.997,0.003,30,31,31,31,1,0,2,2,4,1,1.48,-0.64,0.64,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,SI,2018-10-01,93.1,-1.42,6.19,"Jansen, Kenley",467827,445276,<NA>,called_strike,<NA>,<NA>,<NA>,<NA>,4,Called Strike,R,L,R,LAD,COL,S,<NA>,<NA>,0,0,2018,-0.43,1.4,-0.49627,2.318395,<NA>,<NA>,<NA>,2,9,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,3.430352,-135.644231,-7.572204,-6.20892,25.588025,-12.890645,3.34,1.65,<NA>,<NA>,<NA>,95.4,2243,7.3,570335,518735,641355,571771,457759,592518,592626,621035,624577,53.23,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,71,1,Sinker,5,2,2,5,2,5,2,5,Standard,Standard,197,0.001,-0.041,<NA>,<NA>,<NA>,0.041,<NA>,3,-3,0.996,0.004,30,31,31,31,1,0,2,2,4,1,1.11,0.43,-0.43,<NA>,<NA>,<NA>,<

In [3]:
statcast_dask_data

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,FC,2018-10-01,92.2,-1.97,6.26,"Jansen, Kenley",467827,445276,strikeout,swinging_strike,<NA>,<NA>,<NA>,<NA>,12,Gerardo Parra strikes out swinging.,R,L,R,LAD,COL,S,2,<NA>,0,2,2018,0.39,1.34,0.633962,3.524498,<NA>,<NA>,<NA>,2,9,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,5.848045,-134.451705,-4.262857,3.716224,25.412307,-14.775673,3.32,1.51,<NA>,<NA>,<NA>,94.1,2629,7.0,570335,518735,641355,571771,457759,592518,592626,621035,624577,53.52,<NA>,0.0,0.0,1,0,0,<NA>,71,4,Cutter,5,2,2,5,2,5,2,5,Standard,Standard,164,0.003,-0.153,<NA>,<NA>,<NA>,0.153,<NA>,3,-3,0.997,0.003,30,31,31,31,1,0,2,2,4,1,1.24,-0.39,0.39,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,FC,2018-10-01,93.0,-1.77,6.3,"Jansen, Kenley",467827,445276,<NA>,foul,<NA>,<NA>,<NA>,<NA>,12,Foul,R,L,R,LAD,COL,S,<NA>,<NA>,0,2,2018,0.52,1.26,1.088134,2.441313,<NA>,<NA>,<NA>,2,9,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,6.244121,-135.416081,-7.075826,5.344581,27.081217,-15.009031,3.32,1.51,<NA>,<NA>,<NA>,94.6,2686,7.0,570335,518735,641355,571771,457759,592518,592626,621035,624577,53.5,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,71,3,Cutter,5,2,2,5,2,5,2,5,Standard,Standard,158,0.0,0.0,<NA>,<NA>,<NA>,0.0,<NA>,3,-3,0.997,0.003,30,31,31,31,1,0,2,2,4,1,1.3,-0.52,0.52,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,FC,2018-10-01,91.6,-1.75,6.22,"Jansen, Kenley",467827,445276,<NA>,called_strike,<NA>,<NA>,<NA>,<NA>,5,Called Strike,R,L,R,LAD,COL,S,<NA>,<NA>,0,1,2018,0.64,1.14,-0.172991,2.411909,<NA>,<NA>,<NA>,2,9,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2.602442,-133.526432,-6.436552,7.337142,26.60837,-16.953719,3.36,1.68,<NA>,<NA>,<NA>,93.4,2581,7.1,570335,518735,641355,571771,457759,592518,592626,621035,624577,53.4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,71,2,Cutter,5,2,2,5,2,5,2,5,Standard,Standard,151,0.0,-0.065,<NA>,<NA>,<NA>,0.065,<NA>,3,-3,0.997,0.003,30,31,31,31,1,0,2,2,4,1,1.48,-0.64,0.64,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,SI,2018-10-01,93.1,-1.42,6.19,"Jansen, Kenley",467827,445276,<NA>,called_strike,<NA>,<NA>,<NA>,<NA>,4,Called Strike,R,L,R,LAD,COL,S,<NA>,<NA>,0,0,2018,-0.43,1.4,-0.49627,2.318395,<NA>,<NA>,<NA>,2,9,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,3.430352,-135.644231,-7.572204,-6.20892,25.588025,-12.890645,3.34,1.65,<NA>,<NA>,<NA>,95.4,2243,7.3,570335,518735,641355,571771,457759,592518,592626,621035,624577,53.23,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,71,1,Sinker,5,2,2,5,2,5,2,5,Standard,Standard,197,0.001,-0.041,<NA>,<NA>,<NA>,0.041,<NA>,3,-3,0.996,0.004,30,31,31,31,1,0,2,2,4,1,1.11,0.43,-0.43,<NA>,<NA>,<NA>,<